# AthenaGov AI — Pipeline Mestre & Dashboard de Status

Este notebook é o ponto de **comunicação entre etapas**: cada módulo escreve seu
próprio `status/<modulo>.json` e seu próprio `notebooks/<modulo>_dev_log.ipynb`;
aqui agregamos tudo, mostramos o estado real do projeto, e (quando os módulos
de V1 existirem) executamos o pipeline ponta a ponta:

`Regulatory RAG -> PII Detection -> Policy Engine -> Prompt Security -> Trust Score -> RIPD Engine -> Governance Copilot`

Reexecute este notebook após cada onda de agentes para atualizar o retrato do projeto.

In [ ]:
import json
import pathlib
import pandas as pd

ROOT = pathlib.Path("..").resolve()
STATUS_DIR = ROOT / "status"

rows = []
for f in sorted(STATUS_DIR.glob("*.json")):
    rows.append(json.loads(f.read_text(encoding="utf-8")))

df = pd.DataFrame(rows)
df

## Resumo por status

In [ ]:
if not df.empty:
    display(df.groupby("status")["module"].apply(list))
    print(f"Testes: {df['tests_passed'].sum()}/{df['tests_total'].sum()} passando")
else:
    print("Nenhum status registrado ainda — módulos em construção.")

## Pipeline ponta a ponta

Cada agente de módulo expõe uma função pública simples, documentada no próprio dev-log notebook. Com `core/governance_copilot` completo, o pipeline ponta a ponta agora passa pela API real (FastAPI) em vez de importar `core/ripd_engine` diretamente — é o mesmo caminho que `apps/dashboard` usa em produção, via `GovernanceCopilotClient`.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# Consolidação V1: os 10 itens do ROADMAP estão prontos, incluindo
# core.governance_copilot (item 9). Demonstração ponta a ponta chamando a
# aplicação FastAPI real (em processo, via TestClient) para gerar um RIPD
# completo -- exatamente o mesmo caminho que apps/dashboard usa em produção
# via GovernanceCopilotClient, só que sobre a app em processo em vez de HTTP
# de rede.
from fastapi.testclient import TestClient
from core.governance_copilot.api import app
from core.regulatory_rag.index import build_index, DEFAULT_DATA_DIR

has_index = DEFAULT_DATA_DIR.exists() and any(DEFAULT_DATA_DIR.glob("*.sqlite3"))
if not has_index:
    build_index()

client = TestClient(app)
r = client.post("/api/v1/ripd/generate", json={
    "project_name": "Assistente de triagem de e-mails de ouvidoria",
    "project_description": "Classifica e-mails recebidos pela ouvidoria por urgência e tema.",
    "data_categories": ["personal"],
    "legal_basis": "legal_obligation",
    "context": {"automated_decision": False},
})
print(f"POST /api/v1/ripd/generate (via core.governance_copilot.api) -> {r.status_code}")
report = r.json()
print(f"Projeto: {report['project_name']}")
print(f"Nível de risco: {report['trust_score']['risk_level']} ({report['trust_score']['score']}/100)")
print(f"Decisões de política: {[(d['policy_id'], d['status']) for d in report['policy_decisions']]}")
print()
print("--- Resumo executivo (RIPDReport.executive_summary) ---")
print(report["executive_summary"])

POST /api/v1/ripd/generate (via core.governance_copilot.api) -> 200
Projeto: Assistente de triagem de e-mails de ouvidoria
Nível de risco: low (100.0/100)
Decisões de política: [('POL-009', 'allow')]

--- Resumo executivo (RIPDReport.executive_summary) ---
RIPD do projeto 'Assistente de triagem de e-mails de ouvidoria': nível de risco final classificado como BAIXO (low), AI Trust Score 100.0/100. Decisões de política mais críticas: POL-009 — PERMITIDA (risco baixo). Nenhum dado pessoal ou sensível foi identificado na descrição do projeto submetida a este RIPD. Contexto regulatório da LGPD consultado para este RIPD: 37º, 7º, 11º.
